# NYC Mobility - Bronze Incremental Load

## Goal

We first load March, then add April and May without rebuilding March. Before loading a file, we check whether the same source file is already present. Run `02_bronze_setup` first.


## March initial load

The first batch reads the March Parquet file and adds only ingestion metadata.


In [0]:
-- load march only if this source file is not already in bronze

INSERT INTO `ftw-week-08`.`01_bronze`.`green_taxi`

SELECT
    src.*,
    'nyc_tlc' AS source_system,
    'green_tripdata_2026-03.parquet' AS source_file,
    'green_taxi_2026_03' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at

FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
    format => 'parquet'
) AS src

WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    WHERE source_file = 'green_tripdata_2026-03.parquet'
);

The original run inserted **44,208 rows**. We confirm the file-level count before recording success.


In [0]:
-- validate march source-to-bronze row preservation

SELECT
    source_file,
    batch_id,
    COUNT(*) AS bronze_row_count
FROM `ftw-week-08`.`01_bronze`.`green_taxi`
WHERE source_file = 'green_tripdata_2026-03.parquet'
GROUP BY
    source_file,
    batch_id;

The total after the initial March load is 44,208 rows.


In [0]:
SELECT COUNT(*) AS total_bronze_rows
FROM `ftw-week-08`.`01_bronze`.`green_taxi`;

## Record March success

The MERGE writes one receipt only when the same source identifier and batch ID are not already logged.


In [0]:
-- record the successfully validated march batch

MERGE INTO `ftw-week-08`.`01_bronze`.`ingestion_log` AS target

USING (
    SELECT
        'nyc_tlc' AS source_system,
        'green_taxi' AS source_name,
        'parquet' AS source_type,
        'green_tripdata_2026-03.parquet' AS source_identifier,
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet' AS source_path,
        'green_taxi_2026_03' AS batch_id,
        'SUCCESS' AS status,
        COUNT(*) AS rows_loaded,
        MAX(ingested_at) AS ingested_at
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    WHERE source_file = 'green_tripdata_2026-03.parquet'
) AS incoming

ON target.source_identifier = incoming.source_identifier
AND target.batch_id = incoming.batch_id

WHEN NOT MATCHED THEN
INSERT (
    source_system,
    source_name,
    source_type,
    source_identifier,
    source_path,
    batch_id,
    status,
    rows_loaded,
    ingested_at
)
VALUES (
    incoming.source_system,
    incoming.source_name,
    incoming.source_type,
    incoming.source_identifier,
    incoming.source_path,
    incoming.batch_id,
    incoming.status,
    incoming.rows_loaded,
    incoming.ingested_at
);

The original log output shows one successful March entry with 44,208 rows.


In [0]:
SELECT *
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
ORDER BY ingested_at;

## March idempotency check

We deliberately run the same March INSERT again. The file-level check prevents a second copy.


In [0]:
-- load march only if this source file is not already in bronze

INSERT INTO `ftw-week-08`.`01_bronze`.`green_taxi`

SELECT
    src.*,
    'nyc_tlc' AS source_system,
    'green_tripdata_2026-03.parquet' AS source_file,
    'green_taxi_2026_03' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at

FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
    format => 'parquet'
) AS src

WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    WHERE source_file = 'green_tripdata_2026-03.parquet'
);

The rerun inserted **0 rows**, so the March batch remained unchanged. The next check confirms that it also remained a single log record.


In [0]:
SELECT COUNT(*) AS march_log_rows
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_identifier = 'green_tripdata_2026-03.parquet';

## April incremental load

We add April only. March stays untouched. The accidental duplicate April cell from the original notebook is not repeated here.


In [0]:
-- load april only if this source file is not already in bronze

INSERT INTO `ftw-week-08`.`01_bronze`.`green_taxi`

SELECT
    src.*,
    'nyc_tlc' AS source_system,
    'green_tripdata_2026-04.parquet' AS source_file,
    'green_taxi_2026_04' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at

FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-04.parquet',
    format => 'parquet'
) AS src

WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    WHERE source_file = 'green_tripdata_2026-04.parquet'
);

The original April load inserted **44,238 rows**.


In [0]:
SELECT
    source_file,
    batch_id,
    COUNT(*) AS april_bronze_rows
FROM `ftw-week-08`.`01_bronze`.`green_taxi`
WHERE source_file = 'green_tripdata_2026-04.parquet'
GROUP BY
    source_file,
    batch_id;

After April, the cumulative Bronze count is 88,446 rows.


In [0]:
SELECT COUNT(*) AS total_bronze_rows
FROM `ftw-week-08`.`01_bronze`.`green_taxi`;

We record the validated April batch in the ingestion log.


In [0]:
-- record the successfully validated april batch

MERGE INTO `ftw-week-08`.`01_bronze`.`ingestion_log` AS target

USING (
    SELECT
        'nyc_tlc' AS source_system,
        'green_taxi' AS source_name,
        'parquet' AS source_type,
        'green_tripdata_2026-04.parquet' AS source_identifier,
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-04.parquet' AS source_path,
        'green_taxi_2026_04' AS batch_id,
        'SUCCESS' AS status,
        COUNT(*) AS rows_loaded,
        MAX(ingested_at) AS ingested_at
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    WHERE source_file = 'green_tripdata_2026-04.parquet'
) AS incoming

ON target.source_identifier = incoming.source_identifier
AND target.batch_id = incoming.batch_id

WHEN NOT MATCHED THEN
INSERT (
    source_system,
    source_name,
    source_type,
    source_identifier,
    source_path,
    batch_id,
    status,
    rows_loaded,
    ingested_at
)
VALUES (
    incoming.source_system,
    incoming.source_name,
    incoming.source_type,
    incoming.source_identifier,
    incoming.source_path,
    incoming.batch_id,
    incoming.status,
    incoming.rows_loaded,
    incoming.ingested_at
);

The progressive log output now contains March and April.


In [0]:
SELECT
    source_identifier,
    batch_id,
    status,
    rows_loaded,
    ingested_at
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_name = 'green_taxi'
ORDER BY ingested_at;

## May incremental load

We add the May file without rebuilding the earlier months.


In [0]:
-- load may only if this source file is not already in bronze

INSERT INTO `ftw-week-08`.`01_bronze`.`green_taxi`

SELECT
    src.*,
    'nyc_tlc' AS source_system,
    'green_tripdata_2026-05.parquet' AS source_file,
    'green_taxi_2026_05' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at

FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-05.parquet',
    format => 'parquet'
) AS src

WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    WHERE source_file = 'green_tripdata_2026-05.parquet'
);

The original May load inserted **44,921 rows**.


In [0]:
-- validate may source-to-bronze row preservation

SELECT
    source_file,
    batch_id,
    COUNT(*) AS may_bronze_rows
FROM `ftw-week-08`.`01_bronze`.`green_taxi`
WHERE source_file = 'green_tripdata_2026-05.parquet'
GROUP BY
    source_file,
    batch_id;

The cumulative Bronze count is now **133,367 rows**.


In [0]:
SELECT COUNT(*) AS total_bronze_rows
FROM `ftw-week-08`.`01_bronze`.`green_taxi`;

We record the validated May batch in the ingestion log.


In [0]:
-- record the successfully validated may batch

MERGE INTO `ftw-week-08`.`01_bronze`.`ingestion_log` AS target

USING (
    SELECT
        'nyc_tlc' AS source_system,
        'green_taxi' AS source_name,
        'parquet' AS source_type,
        'green_tripdata_2026-05.parquet' AS source_identifier,
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-05.parquet' AS source_path,
        'green_taxi_2026_05' AS batch_id,
        'SUCCESS' AS status,
        COUNT(*) AS rows_loaded,
        MAX(ingested_at) AS ingested_at
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    WHERE source_file = 'green_tripdata_2026-05.parquet'
) AS incoming

ON target.source_identifier = incoming.source_identifier
AND target.batch_id = incoming.batch_id

WHEN NOT MATCHED THEN
INSERT (
    source_system,
    source_name,
    source_type,
    source_identifier,
    source_path,
    batch_id,
    status,
    rows_loaded,
    ingested_at
)
VALUES (
    incoming.source_system,
    incoming.source_name,
    incoming.source_type,
    incoming.source_identifier,
    incoming.source_path,
    incoming.batch_id,
    incoming.status,
    incoming.rows_loaded,
    incoming.ingested_at
);

## May idempotency test

We deliberately run May a second time to test the same file-level guard.


In [0]:
-- rerun may to prove idempotency

INSERT INTO `ftw-week-08`.`01_bronze`.`green_taxi`

SELECT
    src.*,
    'nyc_tlc' AS source_system,
    'green_tripdata_2026-05.parquet' AS source_file,
    'green_taxi_2026_05' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at

FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-05.parquet',
    format => 'parquet'
) AS src

WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    WHERE source_file = 'green_tripdata_2026-05.parquet'
);

The second May run inserted **0 rows**, so the same batch was not duplicated.

## Green Taxi Load Summary

| Batch | Rows loaded |
|---|---:|
| March 2026 | 44,208 |
| April 2026 | 44,238 |
| May 2026 | 44,921 |
| **Total** | **133,367** |

These are preserved, executed results from the original work. The following Taxi Zones, Weather, and Traffic Advisory cells are new and must be run in Databricks before their results are reported.


## Taxi Zones Bronze load

We load the exact CSV once. Because the source is already tabular, its 265 rows remain row-shaped and unchanged apart from provenance metadata.


In [0]:
-- load the exact taxi zone file only once

INSERT INTO `ftw-week-08`.`01_bronze`.`taxi_zones`
SELECT
    src.*,
    'nyc_tlc' AS source_system,
    'taxi_zone_lookup.csv' AS source_file,
    'taxi_zone_lookup.csv' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/taxi_zones/taxi_zone_lookup.csv',
    format => 'csv',
    header => true
) AS src
WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`taxi_zones`
    WHERE source_file = 'taxi_zone_lookup.csv'
);


### Validate before logging

Expected after a successful Databricks run: `source_row_count = 265`, `bronze_row_count = 265`, and `counts_match = true`. This expected result is not recorded as observed until the cell is executed.


In [0]:
-- validate the taxi zone batch before recording success

WITH source_count AS (
    SELECT COUNT(*) AS row_count
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/taxi_zones/taxi_zone_lookup.csv',
        format => 'csv',
        header => true
    )
),
bronze_count AS (
    SELECT COUNT(*) AS row_count
    FROM `ftw-week-08`.`01_bronze`.`taxi_zones`
    WHERE source_file = 'taxi_zone_lookup.csv'
)
SELECT
    source_count.row_count AS source_row_count,
    bronze_count.row_count AS bronze_row_count,
    source_count.row_count = bronze_count.row_count AS counts_match
FROM source_count
CROSS JOIN bronze_count;


We write one SUCCESS receipt only when the table contains the expected 265 rows. The MERGE prevents a duplicate log entry.


In [0]:
-- record taxi zones only after its expected row count is present

MERGE INTO `ftw-week-08`.`01_bronze`.`ingestion_log` AS target
USING (
    SELECT
        'nyc_tlc' AS source_system,
        'taxi_zones' AS source_name,
        'csv' AS source_type,
        'taxi_zone_lookup.csv' AS source_identifier,
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/taxi_zones/taxi_zone_lookup.csv' AS source_path,
        'taxi_zone_lookup.csv' AS batch_id,
        'SUCCESS' AS status,
        COUNT(*) AS rows_loaded,
        MAX(ingested_at) AS ingested_at
    FROM `ftw-week-08`.`01_bronze`.`taxi_zones`
    WHERE source_file = 'taxi_zone_lookup.csv'
    HAVING COUNT(*) = 265
) AS incoming
ON target.source_identifier = incoming.source_identifier
AND target.batch_id = incoming.batch_id
WHEN NOT MATCHED THEN
INSERT (
    source_system,
    source_name,
    source_type,
    source_identifier,
    source_path,
    batch_id,
    status,
    rows_loaded,
    ingested_at
)
VALUES (
    incoming.source_system,
    incoming.source_name,
    incoming.source_type,
    incoming.source_identifier,
    incoming.source_path,
    incoming.batch_id,
    incoming.status,
    incoming.rows_loaded,
    incoming.ingested_at
);


### Taxi Zones idempotency test

Rerun the same INSERT below. Expected command result: **0 rows inserted**. The post-rerun count should remain **265**.


In [0]:
-- load the exact taxi zone file only once

INSERT INTO `ftw-week-08`.`01_bronze`.`taxi_zones`
SELECT
    src.*,
    'nyc_tlc' AS source_system,
    'taxi_zone_lookup.csv' AS source_file,
    'taxi_zone_lookup.csv' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/taxi_zones/taxi_zone_lookup.csv',
    format => 'csv',
    header => true
) AS src
WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`taxi_zones`
    WHERE source_file = 'taxi_zone_lookup.csv'
);


## Weather raw Bronze load

Databricks `binaryFile` reads the saved JSON as one file record. We cast its full binary content to `raw_json`; we do not create 2,208 hourly Bronze rows.


In [0]:
-- load one complete raw json record for the exact saved weather batch

INSERT INTO `ftw-week-08`.`01_bronze`.`weather_raw`
SELECT
    CAST(src.content AS STRING) AS raw_json,
    'open_meteo' AS source_system,
    'https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.006&start_date=2026-03-01&end_date=2026-05-31&hourly=temperature_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m&timezone=America%2FNew_York' AS source_url,
    'open_meteo_2026-03-01_2026-05-31.json' AS source_file,
    'open_meteo_2026-03-01_2026-05-31.json' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/weather/open_meteo_2026-03-01_2026-05-31.json',
    format => 'binaryFile'
) AS src
WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`weather_raw`
    WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
);


### Validate before logging

Expected after execution: one raw batch record, a non-null `raw_json`, and complete provenance. The 2,208 hourly observations are still inside that JSON.


In [0]:
-- validate the one-record weather raw batch before logging success

SELECT
    COUNT(*) AS raw_batch_count,
    SUM(CASE WHEN raw_json IS NULL OR LENGTH(raw_json) = 0 THEN 1 ELSE 0 END) AS null_or_empty_raw_json,
    SUM(
        CASE
            WHEN source_system IS NULL
              OR source_url IS NULL
              OR source_file IS NULL
              OR batch_id IS NULL
              OR ingested_at IS NULL
            THEN 1 ELSE 0
        END
    ) AS rows_with_missing_provenance
FROM `ftw-week-08`.`01_bronze`.`weather_raw`
WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json';


The SUCCESS log is added only when exactly one complete raw JSON record exists for this filename.


In [0]:
-- record weather only after its raw batch is complete

MERGE INTO `ftw-week-08`.`01_bronze`.`ingestion_log` AS target
USING (
    SELECT
        'open_meteo' AS source_system,
        'weather' AS source_name,
        'json' AS source_type,
        'open_meteo_2026-03-01_2026-05-31.json' AS source_identifier,
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/weather/open_meteo_2026-03-01_2026-05-31.json' AS source_path,
        'open_meteo_2026-03-01_2026-05-31.json' AS batch_id,
        'SUCCESS' AS status,
        COUNT(*) AS rows_loaded,
        MAX(ingested_at) AS ingested_at
    FROM `ftw-week-08`.`01_bronze`.`weather_raw`
    WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    HAVING COUNT(*) = 1
       AND SUM(CASE WHEN raw_json IS NULL OR LENGTH(raw_json) = 0 THEN 1 ELSE 0 END) = 0
) AS incoming
ON target.source_identifier = incoming.source_identifier
AND target.batch_id = incoming.batch_id
WHEN NOT MATCHED THEN
INSERT (
    source_system,
    source_name,
    source_type,
    source_identifier,
    source_path,
    batch_id,
    status,
    rows_loaded,
    ingested_at
)
VALUES (
    incoming.source_system,
    incoming.source_name,
    incoming.source_type,
    incoming.source_identifier,
    incoming.source_path,
    incoming.batch_id,
    incoming.status,
    incoming.rows_loaded,
    incoming.ingested_at
);


### Weather idempotency test

Rerun the load against the same JSON file. Expected command result: **0 rows inserted**. The post-rerun raw batch count should remain **1**.


In [0]:
-- load one complete raw json record for the exact saved weather batch

INSERT INTO `ftw-week-08`.`01_bronze`.`weather_raw`
SELECT
    CAST(src.content AS STRING) AS raw_json,
    'open_meteo' AS source_system,
    'https://archive-api.open-meteo.com/v1/archive?latitude=40.7128&longitude=-74.006&start_date=2026-03-01&end_date=2026-05-31&hourly=temperature_2m,precipitation,rain,snowfall,weather_code,wind_speed_10m&timezone=America%2FNew_York' AS source_url,
    'open_meteo_2026-03-01_2026-05-31.json' AS source_file,
    'open_meteo_2026-03-01_2026-05-31.json' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/weather/open_meteo_2026-03-01_2026-05-31.json',
    format => 'binaryFile'
) AS src
WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`weather_raw`
    WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
);


## Traffic Advisory raw Bronze load — bonus

We combine the exact saved HTML and metadata JSON into one raw batch record. We do not parse its seven headings, events, or roads.


In [0]:
-- load the same saved html and metadata files as one raw scrape batch

INSERT INTO `ftw-week-08`.`01_bronze`.`traffic_advisory_raw`
SELECT
    CAST(html.content AS STRING) AS raw_html,
    CAST(meta.content AS STRING) AS raw_metadata_json,
    'nyc_dot' AS source_system,
    'https://www.nyc.gov/html/dot/html/motorist/wkndtraf.shtml' AS source_url,
    'nyc_dot_weekend_traffic_20260914T040846Z.html' AS source_file,
    '20260914T040846Z' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/traffic_advisory/nyc_dot_weekend_traffic_20260914T040846Z.html',
    format => 'binaryFile'
) AS html
CROSS JOIN read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/traffic_advisory/nyc_dot_weekend_traffic_20260914T040846Z.metadata.json',
    format => 'binaryFile'
) AS meta
WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`traffic_advisory_raw`
    WHERE source_file = 'nyc_dot_weekend_traffic_20260914T040846Z.html'
      AND batch_id = '20260914T040846Z'
);


### Validate before logging

Expected after execution: one raw scrape batch with non-empty `raw_html`, non-empty `raw_metadata_json`, and complete provenance.


In [0]:
-- validate the one-record traffic raw batch before logging success

SELECT
    COUNT(*) AS raw_batch_count,
    SUM(CASE WHEN raw_html IS NULL OR LENGTH(raw_html) = 0 THEN 1 ELSE 0 END) AS null_or_empty_raw_html,
    SUM(CASE WHEN raw_metadata_json IS NULL OR LENGTH(raw_metadata_json) = 0 THEN 1 ELSE 0 END) AS null_or_empty_raw_metadata_json,
    SUM(
        CASE
            WHEN source_system IS NULL
              OR source_url IS NULL
              OR source_file IS NULL
              OR batch_id IS NULL
              OR ingested_at IS NULL
            THEN 1 ELSE 0
        END
    ) AS rows_with_missing_provenance
FROM `ftw-week-08`.`01_bronze`.`traffic_advisory_raw`
WHERE source_file = 'nyc_dot_weekend_traffic_20260914T040846Z.html'
  AND batch_id = '20260914T040846Z';


The SUCCESS receipt is written only for one complete saved scrape batch. This remains bonus evidence and does not control the core gate.


In [0]:
-- record traffic only after both raw payloads are complete

MERGE INTO `ftw-week-08`.`01_bronze`.`ingestion_log` AS target
USING (
    SELECT
        'nyc_dot' AS source_system,
        'traffic_advisory' AS source_name,
        'html' AS source_type,
        'nyc_dot_weekend_traffic_20260914T040846Z.html' AS source_identifier,
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/traffic_advisory/nyc_dot_weekend_traffic_20260914T040846Z.html' AS source_path,
        '20260914T040846Z' AS batch_id,
        'SUCCESS' AS status,
        COUNT(*) AS rows_loaded,
        MAX(ingested_at) AS ingested_at
    FROM `ftw-week-08`.`01_bronze`.`traffic_advisory_raw`
    WHERE source_file = 'nyc_dot_weekend_traffic_20260914T040846Z.html'
      AND batch_id = '20260914T040846Z'
    HAVING COUNT(*) = 1
       AND SUM(CASE WHEN raw_html IS NULL OR LENGTH(raw_html) = 0 THEN 1 ELSE 0 END) = 0
       AND SUM(CASE WHEN raw_metadata_json IS NULL OR LENGTH(raw_metadata_json) = 0 THEN 1 ELSE 0 END) = 0
) AS incoming
ON target.source_identifier = incoming.source_identifier
AND target.batch_id = incoming.batch_id
WHEN NOT MATCHED THEN
INSERT (
    source_system,
    source_name,
    source_type,
    source_identifier,
    source_path,
    batch_id,
    status,
    rows_loaded,
    ingested_at
)
VALUES (
    incoming.source_system,
    incoming.source_name,
    incoming.source_type,
    incoming.source_identifier,
    incoming.source_path,
    incoming.batch_id,
    incoming.status,
    incoming.rows_loaded,
    incoming.ingested_at
);


### Traffic Advisory idempotency test

Rerun this Bronze INSERT against the same saved HTML and metadata files. Expected command result: **0 rows inserted**. Do not rerun the scraper, because that would generate a new timestamped batch.


In [0]:
-- load the same saved html and metadata files as one raw scrape batch

INSERT INTO `ftw-week-08`.`01_bronze`.`traffic_advisory_raw`
SELECT
    CAST(html.content AS STRING) AS raw_html,
    CAST(meta.content AS STRING) AS raw_metadata_json,
    'nyc_dot' AS source_system,
    'https://www.nyc.gov/html/dot/html/motorist/wkndtraf.shtml' AS source_url,
    'nyc_dot_weekend_traffic_20260914T040846Z.html' AS source_file,
    '20260914T040846Z' AS batch_id,
    CURRENT_TIMESTAMP() AS ingested_at
FROM read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/traffic_advisory/nyc_dot_weekend_traffic_20260914T040846Z.html',
    format => 'binaryFile'
) AS html
CROSS JOIN read_files(
    '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/traffic_advisory/nyc_dot_weekend_traffic_20260914T040846Z.metadata.json',
    format => 'binaryFile'
) AS meta
WHERE NOT EXISTS (
    SELECT 1
    FROM `ftw-week-08`.`01_bronze`.`traffic_advisory_raw`
    WHERE source_file = 'nyc_dot_weekend_traffic_20260914T040846Z.html'
      AND batch_id = '20260914T040846Z'
);


## Bronze Load Status

The Green Taxi results above are verified. The three newly prepared loads and their reruns have empty outputs by design. Run them in Databricks, then use `04_bronze_validation` before declaring the core Bronze layer complete. We stop before Silver.
